# 🎙️ Kokoro TTS Studio — 8 Preset Voices, 24 kHz (Colab T4 Free)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SudhirDevOps1/t4-gpu-llm-api/blob/main/colab/02_zero_shot_voice_tts.ipynb)
[![Notebook Store](https://img.shields.io/badge/Notebook_Store-Obsidian_Hub-10b981?style=for-the-badge&logo=github)](https://github.com/SudhirDevOps1/t4-gpu-llm-api)

> **What this notebook actually is:** [Kokoro-82M](https://huggingface.co/hexgrad/Kokoro-82M), a fast
> Apache-2.0 text-to-speech model with **8 built-in preset voices**. It does **not** clone your voice
> and it does **not** use Fish Audio. For zero-shot voice cloning, use
> [`08_f5_chatterbox_hindi_voice.ipynb`](08_f5_chatterbox_hindi_voice.ipynb).

- **VRAM:** ~2.8 GB (runs on CPU too) · **Output:** 24 kHz mono WAV
- **Voices:** `af_*` / `am_*` = American English (`lang_code="a"`), `bf_*` / `bm_*` = British English (`lang_code="b"`)
- This notebook picks the matching pipeline automatically, so every voice in the list works.
- The Gradio link is password protected (Basic auth).


In [ ]:
#@title 🚀 1. Hardware Check & Session Fingerprint
import hashlib, os, socket, time, uuid

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

raw_fp = f"colab-tts-{socket.gethostname()}-{uuid.uuid4()}-{time.time()}"
INSTANCE_ID = "colab-tts-" + hashlib.sha256(raw_fp.encode()).hexdigest()[:8]

print(f"🆔 Unique Instance ID: {INSTANCE_ID}")
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv


In [ ]:
#@title 📦 2. Install Kokoro TTS Engine
# espeak-ng is required by Kokoro's phonemizer; the PyPI `kokoro` wheel avoids a source build.
!apt-get -qq update && apt-get -qq -y install espeak-ng > /dev/null 2>&1
!pip install -q --upgrade pip
!pip install -q kokoro soundfile gradio
print("✅ Kokoro TTS installed.")


In [ ]:
#@title 🎙️ 3. Launch Kokoro Studio (password protected)
import os
import time

import gradio as gr
import soundfile as sf
import torch
from kokoro import KPipeline

OUTPUT_DIR = "/content" if os.path.isdir("/content") else "/tmp"
VOICES = [
    "af_heart", "af_bella", "af_nicole", "af_sky",
    "am_adam", "am_michael",
    "bf_emma", "bm_george",
]

_PIPELINES = {}


def pipeline_for(voice: str) -> KPipeline:
    """Kokoro needs lang_code 'a' (US) or 'b' (UK); the voice prefix selects it."""
    lang_code = voice[0]
    if lang_code not in _PIPELINES:
        print(f"⚡ Loading Kokoro pipeline (lang_code={lang_code!r})...")
        _PIPELINES[lang_code] = KPipeline(lang_code=lang_code)
    return _PIPELINES[lang_code]


def generate_speech(text: str, voice: str, speed: float):
    if not text.strip():
        raise gr.Error("Please enter some text first.")
    pipeline = pipeline_for(voice)
    segments = [audio for _, _, audio in pipeline(text, voice=voice, speed=float(speed))]
    if not segments:
        raise gr.Error("Kokoro returned no audio for this input.")
    audio = torch.cat([seg if torch.is_tensor(seg) else torch.as_tensor(seg) for seg in segments]).float().numpy()
    out_path = os.path.join(OUTPUT_DIR, f"kokoro_{int(time.time())}.wav")
    sf.write(out_path, audio, 24000)
    return out_path


def studio_auth():
    """Password for the public Gradio link (Basic auth).

    Priority: STUDIO_PASSWORD -> Colab secret -> Kaggle secret -> random one-time password.
    """
    import os
    import secrets

    password = os.getenv("STUDIO_PASSWORD", "")
    if not password:
        try:
            from google.colab import userdata
            password = userdata.get("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        try:
            from kaggle_secrets import UserSecretsClient
            password = UserSecretsClient().get_secret("STUDIO_PASSWORD") or ""
        except Exception:
            password = ""
    if not password:
        password = secrets.token_urlsafe(9)
        print(f"\n🔑 One-time password for this session: {password}\n")
    return ("admin", password)

demo = gr.Interface(
    fn=generate_speech,
    inputs=[
        gr.Textbox(label="Text", lines=4,
                   value="Kokoro runs fast text-to-speech on a free Google Colab Tesla T4."),
        gr.Dropdown(choices=VOICES, value="af_heart", label="Voice (af/am = US, bf/bm = UK)"),
        gr.Slider(minimum=0.5, maximum=2.0, value=1.0, step=0.1, label="Speed"),
    ],
    outputs=gr.Audio(label="Synthesized audio (24 kHz)", type="filepath"),
    title="🎙️ Kokoro TTS Studio — Colab Free T4",
    description="8 preset Kokoro voices, 24 kHz output. This is text-to-speech, not voice cloning.",
)

demo.launch(share=True, auth=studio_auth(), debug=False)
